# Stage 1 MVP: Road Freight Delivery Sequence Optimization

This Google Colab notebook runs the Stage 1 MVP only:

- Single-route, single-vehicle delivery stop sequencing.
- Amazon Last Mile Routing Research Challenge Dataset.
- One baseline: Nearest Neighbour.
- One ML model: Random Forest next-stop prediction.

The notebook expects the dataset files `route_data.json`, `actual_sequences.json`, `package_data.json`, and `travel_times.json`. The full `travel_times.json` is not loaded into memory; only selected route IDs are streamed with `ijson`.

In [1]:
# Colab setup: works both from a checked-out repo and when opened directly from GitHub.
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Chen0421-c/delivery-sequence-ml.git"
REPO_DIR = Path("/content/delivery-sequence-ml")

if not Path("requirements.txt").exists():
    if not REPO_DIR.exists():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
    os.chdir(REPO_DIR)

print("Notebook working directory:", Path.cwd())
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])


Notebook working directory: /content/delivery-sequence-ml


0

In [2]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    # Update this path if you cloned the repository elsewhere in Colab.
    REPO_ROOT = Path("/content/delivery-sequence-ml")

sys.path.insert(0, str(REPO_ROOT))
print("Repository root:", REPO_ROOT)

Repository root: /content/delivery-sequence-ml


## Locate or provide the dataset

Place the Amazon dataset under `data/` or set `DATA_DIR` to a Google Drive folder or uploaded folder that contains the JSON files. The repository `.gitignore` excludes `data/` because the dataset is large.

In [3]:
# Optional Google Drive mount.
# from google.colab import drive
# drive.mount('/content/drive')

DATA_DIR = REPO_ROOT / "data"  # Change this to your dataset folder when needed.
N_ROUTES = 100                 # Stage 1 default small working subset.
RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"

required = ["route_data.json", "actual_sequences.json", "package_data.json", "travel_times.json"]
print("Dataset folder:", DATA_DIR)
print("Expected files:", required)

Dataset folder: /content/delivery-sequence-ml/data
Expected files: ['route_data.json', 'actual_sequences.json', 'package_data.json', 'travel_times.json']


In [5]:
from pathlib import Path
import subprocess

DATA_DIR = Path("/content/delivery-sequence-ml/data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Install AWS CLI if needed
subprocess.run(["pip", "install", "-q", "awscli"], check=True)

# Amazon Last Mile dataset public S3 bucket
S3_BASE = "s3://amazon-last-mile-challenges/almrrc2021/almrrc2021-data-training/model_build_inputs"

files = [
    "route_data.json",
    "actual_sequences.json",
    "package_data.json",
    "travel_times.json",
]

for filename in files:
    local_path = DATA_DIR / filename
    if local_path.exists():
        print(f"Already exists: {local_path}")
    else:
        s3_path = f"{S3_BASE}/{filename}"
        print(f"Downloading {filename}...")
        subprocess.run(
            ["aws", "s3", "cp", "--no-sign-request", s3_path, str(local_path)],
            check=True,
        )

print("Download complete.")
print("Files in data directory:")
for p in DATA_DIR.glob("*.json"):
    print(p.name, round(p.stat().st_size / (1024 ** 2), 2), "MB")

Already exists: /content/delivery-sequence-ml/data/route_data.json
Already exists: /content/delivery-sequence-ml/data/actual_sequences.json
Already exists: /content/delivery-sequence-ml/data/package_data.json
Already exists: /content/delivery-sequence-ml/data/travel_times.json
Download complete.
Files in data directory:
package_data.json 358.05 MB
travel_times.json 1732.97 MB
route_data.json 75.31 MB
actual_sequences.json 9.22 MB


## Memory-safe load and preprocessing

This step streams the top-level route IDs and loads only the selected subset from every dataset file, including `travel_times.json`.

In [7]:
from pathlib import Path
import os
import shutil

RAW_DATA_DIR = Path("/content/delivery-sequence-ml/data")
CLEAN_DATA_DIR = Path("/content/delivery-sequence-ml/data_clean")
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

def file_contains_nan(path, chunk_size=1024 * 1024):
    """Check whether a large file contains the invalid JSON token NaN."""
    with open(path, "rb") as f:
        previous = b""
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                return False
            combined = previous + chunk
            if b"NaN" in combined:
                return True
            previous = combined[-2:]

def replace_nan_with_null(src_path, dst_path, chunk_size=1024 * 1024):
    """
    Create a cleaned JSON file by replacing invalid NaN tokens with null.
    This is done in chunks to avoid loading large files into memory.
    """
    dst_path.parent.mkdir(parents=True, exist_ok=True)

    with open(src_path, "rb") as src, open(dst_path, "wb") as dst:
        carry = b""
        while True:
            chunk = src.read(chunk_size)
            if not chunk:
                if carry:
                    dst.write(carry.replace(b"NaN", b"null"))
                break

            combined = carry + chunk
            safe_part = combined[:-2]
            carry = combined[-2:]
            dst.write(safe_part.replace(b"NaN", b"null"))

required_files = [
    "route_data.json",
    "actual_sequences.json",
    "package_data.json",
    "travel_times.json",
]

for filename in required_files:
    src = RAW_DATA_DIR / filename
    dst = CLEAN_DATA_DIR / filename

    if not src.exists():
        raise FileNotFoundError(f"Missing source file: {src}")

    if dst.exists():
        print(f"Already cleaned or linked: {dst}")
        continue

    print(f"Checking {filename}...")
    if file_contains_nan(src):
        print(f"Found NaN in {filename}. Creating cleaned copy...")
        replace_nan_with_null(src, dst)
    else:
        print(f"No NaN found in {filename}. Creating symlink...")
        try:
            os.symlink(src, dst)
        except FileExistsError:
            pass
        except OSError:
            shutil.copy2(src, dst)

print("\nClean data directory ready:")
for p in CLEAN_DATA_DIR.glob("*.json"):
    size_mb = p.stat().st_size / (1024 ** 2)
    print(p.name, round(size_mb, 2), "MB")

# IMPORTANT: use cleaned data folder from now on
DATA_DIR = CLEAN_DATA_DIR
print("\nDATA_DIR has been changed to:", DATA_DIR)

Checking route_data.json...
Found NaN in route_data.json. Creating cleaned copy...
Checking actual_sequences.json...
No NaN found in actual_sequences.json. Creating symlink...
Checking package_data.json...
Found NaN in package_data.json. Creating cleaned copy...
Checking travel_times.json...
No NaN found in travel_times.json. Creating symlink...

Clean data directory ready:
package_data.json 360.61 MB
travel_times.json 1732.97 MB
route_data.json 75.33 MB
actual_sequences.json 9.22 MB

DATA_DIR has been changed to: /content/delivery-sequence-ml/data_clean


In [8]:
from pathlib import Path

print("Current DATA_DIR:", DATA_DIR)
print("DATA_DIR exists:", Path(DATA_DIR).exists())

for name in ["route_data.json", "actual_sequences.json", "package_data.json", "travel_times.json"]:
    p = Path(DATA_DIR) / name
    print(name, "exists:", p.exists(), "size MB:", round(p.stat().st_size / (1024**2), 2) if p.exists() else None)

Current DATA_DIR: /content/delivery-sequence-ml/data_clean
DATA_DIR exists: True
route_data.json exists: True size MB: 75.33
actual_sequences.json exists: True size MB: 9.22
package_data.json exists: True size MB: 360.61
travel_times.json exists: True size MB: 1732.97


In [9]:
from pathlib import Path

p = Path("/content/delivery-sequence-ml/data_clean/route_data.json")

with open(p, "rb") as f:
    sample = f.read(20_000_000)  # read first 20MB only

print("NaN still exists in cleaned route_data first 20MB:", b"NaN" in sample)

idx = sample.find(b"NaN")
if idx != -1:
    print("Example around NaN:")
    print(sample[max(0, idx-80): idx+80])

NaN still exists in cleaned route_data first 20MB: False


In [10]:
from pathlib import Path
import re

RAW_DATA_DIR = Path("/content/delivery-sequence-ml/data")
CLEAN_DATA_DIR = Path("/content/delivery-sequence-ml/data_clean_v2")
CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)

def clean_nonstandard_json_tokens(src_path, dst_path, chunk_size=1024 * 1024):
    """
    Replace non-standard JSON constants with valid JSON null:
    NaN, Infinity, -Infinity.
    """
    pattern = re.compile(rb'(?<![A-Za-z0-9_."-])(?:NaN|Infinity|-Infinity)(?![A-Za-z0-9_"])')

    with open(src_path, "rb") as src, open(dst_path, "wb") as dst:
        carry = b""
        while True:
            chunk = src.read(chunk_size)
            if not chunk:
                if carry:
                    dst.write(pattern.sub(b"null", carry))
                break

            combined = carry + chunk

            if len(combined) > 64:
                safe_part = combined[:-64]
                carry = combined[-64:]
            else:
                safe_part = b""
                carry = combined

            if safe_part:
                dst.write(pattern.sub(b"null", safe_part))

required_files = [
    "route_data.json",
    "actual_sequences.json",
    "package_data.json",
    "travel_times.json",
]

for filename in required_files:
    src = RAW_DATA_DIR / filename
    dst = CLEAN_DATA_DIR / filename

    if not src.exists():
        raise FileNotFoundError(f"Missing source file: {src}")

    print(f"Cleaning {filename}...")
    clean_nonstandard_json_tokens(src, dst)

print("\nClean v2 data directory ready:")
for p in CLEAN_DATA_DIR.glob("*.json"):
    print(p.name, round(p.stat().st_size / (1024 ** 2), 2), "MB")

DATA_DIR = CLEAN_DATA_DIR
print("\nDATA_DIR has been changed to:", DATA_DIR)

Cleaning route_data.json...
Cleaning actual_sequences.json...
Cleaning package_data.json...
Cleaning travel_times.json...

Clean v2 data directory ready:
package_data.json 360.61 MB
travel_times.json 1732.97 MB
route_data.json 75.33 MB
actual_sequences.json 9.22 MB

DATA_DIR has been changed to: /content/delivery-sequence-ml/data_clean_v2


In [11]:
from pathlib import Path

p = Path("/content/delivery-sequence-ml/data_clean_v2/route_data.json")

found = False
with open(p, "rb") as f:
    while True:
        chunk = f.read(5_000_000)
        if not chunk:
            break
        if b"NaN" in chunk:
            found = True
            break

print("NaN still exists anywhere in route_data.json:", found)

NaN still exists anywhere in route_data.json: False


In [12]:
from src.data_loader import load_stage1_subset
from src.preprocessing import filter_valid_routes, build_stop_feature_table

dataset = load_stage1_subset(DATA_DIR, n_routes=N_ROUTES)
route_data = dataset["route_data"]
actual_sequences = dataset["actual_sequences"]
package_data = dataset["package_data"]
travel_times = dataset["travel_times"]

valid_route_ids, invalid_report = filter_valid_routes(route_data, actual_sequences, package_data, travel_times)
print(f"Loaded routes: {len(route_data)}")
print(f"Valid routes: {len(valid_route_ids)}")
print(f"Invalid routes: {len(invalid_report)}")
display(invalid_report.head())

stop_features = build_stop_feature_table(valid_route_ids, route_data, actual_sequences, package_data)
display(stop_features.head())

Loaded routes: 100
Valid routes: 100
Invalid routes: 0


""


,route_id,stop_id,actual_order,is_depot,lat,lng,zone_id,stop_type,package_count
0,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,VE,0,True,34.007369,-118.143927,None,Station,0
1,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,TG,1,False,34.088467,-118.284521,A-2.2A,Dropoff,7
2,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,GP,2,False,34.088709,-118.284839,A-2.2A,Dropoff,1
3,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,HT,3,False,34.088717,-118.286484,A-2.2A,Dropoff,2
4,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,AG,4,False,34.089727,-118.28553,A-2.1A,Dropoff,2


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


专门运行一个 sample route 的 Nearest Neighbour baseline

In [13]:
from src.baseline import nearest_neighbour_sequence, calculate_sequence_travel_time
from src.preprocessing import get_actual_sequence, get_depot_stop_id, get_route_stops

# Run Nearest Neighbour baseline on one sample valid route
sample_route_id = valid_route_ids[0]

actual_sequence = get_actual_sequence(sample_route_id, actual_sequences)
depot_id = get_depot_stop_id(sample_route_id, route_data, actual_sequences)
stop_ids = list(get_route_stops(sample_route_id, route_data))
matrix = travel_times[sample_route_id]

nn_sequence = nearest_neighbour_sequence(stop_ids, depot_id, matrix)

actual_time = calculate_sequence_travel_time(actual_sequence, matrix)
nn_time = calculate_sequence_travel_time(nn_sequence, matrix)

print("Sample route:", sample_route_id)
print("Number of stops:", len(stop_ids))
print("Depot ID:", depot_id)
print("Actual route travel time:", actual_time)
print("Nearest Neighbour route travel time:", nn_time)
print("Percentage gap vs actual:", ((nn_time - actual_time) / actual_time) * 100 if actual_time else None)

print("\nActual sequence preview:", actual_sequence[:10])
print("Nearest Neighbour sequence preview:", nn_sequence[:10])

Sample route: RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77
Number of stops: 119
Depot ID: VE
Actual route travel time: 7754.1
Nearest Neighbour route travel time: 7691.2
Percentage gap vs actual: -0.8111837608491062

Actual sequence preview: ['VE', 'TG', 'GP', 'HT', 'AG', 'QM', 'SF', 'BY', 'CW', 'VW']
Nearest Neighbour sequence preview: ['VE', 'TG', 'GP', 'HT', 'AG', 'SC', 'IJ', 'XB', 'CW', 'VW']


对于此次测试路线而言，最近邻基准路线的行驶时间略低于历史实际路线的行驶时间。然而，这并不一定意味着该基准路线在实际操作中更具优势，因为真实的驾驶序列可能反映了那些仅通过行驶时间无法完全体现的实际情况限制。

## Train Random Forest with a route-level split

Candidate rows are generated from route sequences, but train/test split is performed by `route_id` to avoid route leakage.

In [14]:
from src.ml_model import (
    build_next_stop_training_samples,
    next_stop_accuracy,
    route_level_train_test_split,
    train_random_forest_model,
)

train_route_ids, test_route_ids = route_level_train_test_split(valid_route_ids, test_size=0.2, random_state=42)
if not test_route_ids:
    test_route_ids = train_route_ids

train_samples = build_next_stop_training_samples(train_route_ids, route_data, actual_sequences, package_data, travel_times)
test_samples = build_next_stop_training_samples(test_route_ids, route_data, actual_sequences, package_data, travel_times)

model = train_random_forest_model(train_samples, random_state=42)
accuracy = next_stop_accuracy(model, test_samples)
print(f"Train routes: {len(train_route_ids)}")
print(f"Test routes: {len(test_route_ids)}")
print(f"Top-1 next-stop prediction accuracy: {accuracy:.4f}")
display(train_samples.head())

Train routes: 80
Test routes: 20
Top-1 next-stop prediction accuracy: 0.6935


,travel_time_current_to_candidate,current_lat,current_lng,candidate_lat,candidate_lng,same_zone,candidate_package_count,route_stop_count,current_position_ratio,route_id,current_stop,candidate_stop,label
0,2397.3,47.937344,-122.244952,47.683784,-122.290503,0.0,3.0,106.0,0.0,RouteID_0016bc70-cb8d-48b0-aa55-8ee50bdcdb59,UX,BP,1
1,2421.6,47.937344,-122.244952,47.685800,-122.290577,0.0,1.0,106.0,0.0,RouteID_0016bc70-cb8d-48b0-aa55-8ee50bdcdb59,UX,FM,0
2,2423.4,47.937344,-122.244952,47.685103,-122.291602,0.0,3.0,106.0,0.0,RouteID_0016bc70-cb8d-48b0-aa55-8ee50bdcdb59,UX,WD,0
3,2454.6,47.937344,-122.244952,47.686274,-122.291638,0.0,1.0,106.0,0.0,RouteID_0016bc70-cb8d-48b0-aa55-8ee50bdcdb59,UX,YK,0
4,2469.8,47.937344,-122.244952,47.685918,-122.289525,0.0,2.0,106.0,0.0,RouteID_0016bc70-cb8d-48b0-aa55-8ee50bdcdb59,UX,HC,0


## Evaluate actual, Nearest Neighbour, and Random Forest sequences

In [15]:
import pandas as pd
from src.baseline import nearest_neighbour_sequence
from src.evaluation import evaluate_sequence
from src.ml_model import generate_rf_sequence
from src.preprocessing import get_actual_sequence, get_depot_stop_id, get_route_stops

rows = []
example_sequences = {}
for route_id in test_route_ids:
    actual_sequence = get_actual_sequence(route_id, actual_sequences)
    depot_id = get_depot_stop_id(route_id, route_data, actual_sequences)
    stop_ids = list(get_route_stops(route_id, route_data))
    matrix = travel_times[route_id]

    nn_sequence = nearest_neighbour_sequence(stop_ids, depot_id, matrix)
    rf_sequence = generate_rf_sequence(route_id, model, route_data, actual_sequences, package_data, travel_times)

    rows.append(evaluate_sequence(route_id, "actual_historical", actual_sequence, actual_sequence, matrix))
    rows.append(evaluate_sequence(route_id, "nearest_neighbour", actual_sequence, nn_sequence, matrix))
    rows.append(evaluate_sequence(route_id, "random_forest", actual_sequence, rf_sequence, matrix))
    example_sequences = {"route_id": route_id, "actual": actual_sequence, "nearest_neighbour": nn_sequence, "random_forest": rf_sequence}

evaluation = pd.DataFrame(rows)
evaluation["next_stop_prediction_accuracy"] = accuracy
summary = evaluation.groupby("method", as_index=False).mean(numeric_only=True)

RESULTS_DIR.mkdir(exist_ok=True, parents=True)
evaluation.to_csv(RESULTS_DIR / "evaluation.csv", index=False)
summary.to_csv(RESULTS_DIR / "evaluation_summary.csv", index=False)

display(evaluation.head())
display(summary)

,route_id,method,total_travel_time,percentage_gap_vs_actual,kendall_tau,spearman,edit_distance,next_stop_prediction_accuracy
0,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,actual_historical,7754.1,0.000000,1.000000,1.000000,0,0.693532
1,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,nearest_neighbour,7691.2,-0.811184,0.263353,0.377902,109,0.693532
2,RouteID_00143bdd-0a6b-49ec-bb35-36593d303e77,random_forest,8459.9,9.102281,-0.020937,0.015147,106,0.693532
3,RouteID_0021a2aa-780f-460d-b09a-f301709e2523,actual_historical,13523.7,0.000000,1.000000,1.000000,0,0.693532
4,RouteID_0021a2aa-780f-460d-b09a-f301709e2523,nearest_neighbour,15055.3,11.325303,-0.254629,-0.506345,142,0.693532


,method,total_travel_time,percentage_gap_vs_actual,kendall_tau,spearman,edit_distance,next_stop_prediction_accuracy
0,actual_historical,10293.315,0.000000,1.000000,1.000000,0.0,0.693532
1,nearest_neighbour,10397.770,-0.137187,0.036164,0.064832,137.6,0.693532
2,random_forest,11265.360,9.135389,0.053016,0.073909,137.4,0.693532


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


In [16]:
# Recalculate percentage gap from mean total travel time for clearer interpretation

summary_fixed = summary.copy()

actual_mean_time = summary_fixed.loc[
    summary_fixed["method"] == "actual_historical",
    "total_travel_time"
].iloc[0]

summary_fixed["percentage_gap_from_mean_time"] = (
    (summary_fixed["total_travel_time"] - actual_mean_time)
    / actual_mean_time
    * 100
)

display(summary_fixed)

,method,total_travel_time,percentage_gap_vs_actual,kendall_tau,spearman,edit_distance,next_stop_prediction_accuracy,percentage_gap_from_mean_time
0,actual_historical,10293.315,0.000000,1.000000,1.000000,0.0,0.693532,0.000000
1,nearest_neighbour,10397.770,-0.137187,0.036164,0.064832,137.6,0.693532,1.014785
2,random_forest,11265.360,9.135389,0.053016,0.073909,137.4,0.693532,9.443459


## Visualize one route with Folium

In [18]:
from IPython.display import display
from src.visualization import plot_route_folium

FIGURES_DIR.mkdir(exist_ok=True, parents=True)
route_id = example_sequences["route_id"]
actual_map = plot_route_folium(route_id, example_sequences["actual"], route_data, FIGURES_DIR / f"{route_id}_actual.html")
plot_route_folium(route_id, example_sequences["nearest_neighbour"], route_data, FIGURES_DIR / f"{route_id}_nearest_neighbour.html")
plot_route_folium(route_id, example_sequences["random_forest"], route_data, FIGURES_DIR / f"{route_id}_random_forest.html")

print("Saved maps to", FIGURES_DIR)
display(actual_map)

Saved maps to /content/delivery-sequence-ml/figures


## Optional: run the packaged pipeline

The modular pipeline writes preprocessing outputs, evaluation CSVs, and maps to `results/` and `figures/`.

In [19]:
# from src.stage1_pipeline import run_stage1
# outputs = run_stage1(DATA_DIR, n_routes=N_ROUTES, results_dir=RESULTS_DIR, figures_dir=FIGURES_DIR)
# display(outputs["summary"])